# sEMG Prosthetic Gesture Classification
## Notebook 11 (Training): Real Leave-One-Subject-Out Cross-Validation (Google Colab Edition)

This notebook actually **runs** 40-fold LOSO training with the current re-tuned CatBoost model
(308 Optuna trials, best trial #151), rather than reading pre-existing checkpoint artifacts.
The existing 40 checkpoints under `outputs/loso_checkpoints/CATBOOST/` were computed with the
**old, under-tuned (3-trial) model** and are preserved unchanged for comparison; this run writes
to a **new** directory, `outputs/loso_checkpoints_v2/CATBOOST/`.

Each fold trains a fresh CatBoost classifier (cloned from the tuned pipeline's hyperparameters)
on 39 of 40 subjects and evaluates on the held-out subject, then checkpoints immediately to that
new directory. Because each fold is an independent file (not a shared database), this is safe
to interrupt and resume: re-running the training cell skips any fold that already has a
checkpoint.

### What to upload to Drive
Same project structure as the Notebook 09 Colab run, plus `src/ml/loso.py`,
`src/ml/fold_metrics.py`, and `models/optimized/CatBoost.pkl` (already produced by the
Notebook 09 re-tuning).

In [ ]:
# ==============================================================
# GOOGLE COLAB SETUP & ENVIRONMENT INITIALIZATION
# ==============================================================
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/semg-prosthetic-gesture-classification'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"Changed working directory to Google Drive: {PROJECT_PATH}")
    else:
        raise FileNotFoundError(
            f"{PROJECT_PATH} not found. Upload/sync the project folder to this path first."
        )
    !pip install -q catboost xgboost lightgbm scikit-learn pyarrow fastparquet
else:
    PROJECT_PATH = str(Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd()))

print(f"IN_COLAB={IN_COLAB} | PROJECT_PATH={PROJECT_PATH}")


In [ ]:
import sys, os, json, time, pickle
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_PATH)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ml.loso import run_loso_validation

outputs_dir = PROJECT_ROOT / "outputs"
tables_dir = outputs_dir / "tables"
reports_dir = outputs_dir / "reports"
models_dir = PROJECT_ROOT / "models" / "optimized"

# NEW checkpoint directory -- the OLD one (outputs/loso_checkpoints/CATBOOST/)
# is left untouched as a reference from the pre-retuning model.
NEW_CHECKPOINT_DIR = outputs_dir / "loso_checkpoints_v2" / "CATBOOST"
NEW_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Writing fold checkpoints to: {NEW_CHECKPOINT_DIR}")

print("Loading full top-50 feature dataset (all 40 subjects)...")
t0 = time.perf_counter()
df = pd.read_parquet(PROJECT_ROOT / "data/final/selected_features_top50.parquet")
print(f"Loaded {df.shape} in {time.perf_counter()-t0:.1f}s")

with open(models_dir / "CatBoost.pkl", "rb") as f:
    tuned_pipeline = pickle.load(f)
print("Tuned CatBoost params:", tuned_pipeline.named_steps["classifier"].get_params())


### Optional: enable GPU acceleration for CatBoost

CatBoost supports `task_type="GPU"`. This is optional (folds are already CPU-fast for
CatBoost's shallow tuned trees) and defaults to **off** for reliability -- if you enable it and
your Colab runtime doesn't actually have a GPU attached, training will raise a clear error
rather than silently falling back, so only set this to `True` if you've selected a GPU runtime
(Runtime -> Change runtime type -> T4 GPU).

In [ ]:
USE_GPU = False  # set True only if you selected a GPU (T4) runtime above

if USE_GPU:
    tuned_pipeline.set_params(classifier__task_type="GPU", classifier__devices="0")
    print("GPU acceleration enabled for CatBoost.")
else:
    print("Running on CPU (default; already fast for this model's shallow tuned trees).")


In [ ]:
# ==============================================================
# RUN REAL 40-FOLD LOSO TRAINING
# Resumable: already-checkpointed folds (in NEW_CHECKPOINT_DIR) are skipped
# automatically if you stop and re-run this cell.
# ==============================================================
t_start = time.perf_counter()
loso_results = run_loso_validation(
    df=df,
    workspace_dir=PROJECT_ROOT,
    model_name="CATBOOST",
    checkpoint_dir=NEW_CHECKPOINT_DIR,
    force_rerun=False,
)
total_time = time.perf_counter() - t_start
print(f"\nLOSO training complete: {loso_results['n_folds']} folds in {total_time/60:.1f} minutes")

# Quick summary
fold_df = pd.DataFrame(loso_results["fold_metrics"])
print(f"Mean Accuracy: {fold_df['Accuracy'].mean():.4f} +/- {fold_df['Accuracy'].std():.4f}")
print(f"Mean Macro F1: {fold_df['Macro F1'].mean():.4f} +/- {fold_df['Macro F1'].std():.4f}")
fold_df.to_csv(NEW_CHECKPOINT_DIR / "fold_summary.csv", index=False)


### After completion

Download the entire `outputs/loso_checkpoints_v2/CATBOOST/` directory back to the local
project (same relative path). The next step will be rebuilding
`notebooks/11_cross_subject_generalization_LOSO.ipynb` to aggregate and report on these new,
real checkpoints (following the same pattern already used for Notebooks 10, 12, and 14),
replacing its current read-only dependency on the old checkpoint directory.